# Lab 1 - Fundamentals of Machine Learning and systematic experimentation
D7041E Applied Artificial Intelligence, LTU

The functions are in the `lab1/` folder next to this notebook (one file per part of the lab),
so the notebook itself just calls them and shows the results.

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
from matplotlib import pyplot as plt

from lab1 import config, image_dataset, preprocessing, multidigit_mnist, mnist_data
from lab1 import distances, metrics, plotting, cross_validation, svm_iris, uci_experiments
from lab1.nearest_neighbour import NearestNeighbor

pd.set_option("display.width", 160)

# Part 1 - Warming up
## Task 1.1 - Import my own dataset

The photos are in `my_dataset/<category>/` (5 categories, at least 5 photos each). Every photo is
converted to RGB and resized to the same size so they can all be stacked into one (n, x, y, 3) array.

In [ ]:
try:
    X_photos, y_photos, class_names = image_dataset.load_image_folder()
    print(image_dataset.describe_dataset(X_photos, y_photos, class_names))
except FileNotFoundError as err:
    X_photos = None
    print("Photos not found yet ->", err)

### 1.1.2 `PlotSample(index)` - plot one sample with matplotlib

In [ ]:
if X_photos is not None:
    plotting.PlotSample(X_photos, 0, y_photos, class_names)
    fig = plotting.plot_samples_grid(X_photos, y_photos, class_names, indices=range(0, len(X_photos), 5))
    plotting.save(fig, "task1_1_samples")
    plt.show()

### 1.1.3 Pre-processing (following *Preprocessing for deep learning*)

Same three parts as the tutorial:
1. Covariance, computed by hand and checked against `np.cov`.
2. On 2-D toy data: center, standardize, decorrelate, whiten.
3. ZCA whitening of my photos: flatten, scale to 0..1, subtract the mean per pixel, SVD of the
   covariance, apply the ZCA matrix.

In [ ]:
# Part 1 of the tutorial: covariance of uncorrelated and correlated data
A = preprocessing.simulate_uncorrelated()
B = preprocessing.simulate_correlated()
plotting.plotDataAndCov(A, "(uncorrelated)")
plotting.plotDataAndCov(B, "(correlated)")
print("calculateCovariance equals np.cov:", np.allclose(preprocessing.calculateCovariance(B), np.cov(B, rowvar=False, bias=True)))
plt.show()

In [ ]:
# Part 2 of the tutorial: center -> standardize -> decorrelate -> whiten (on the correlated data)
for name, fn in [("centered", preprocessing.center),
                 ("standardized", preprocessing.standardize),
                 ("decorrelated", preprocessing.decorrelate),
                 ("whitened", preprocessing.whiten)]:
    plotting.plotDataAndCov(fn(B), f"({name})")
plt.show()

In [ ]:
# Part 3 of the tutorial: ZCA whitening of my own photos
if X_photos is not None:
    steps = preprocessing.preprocess_my_images(X_photos)
    for key, arr in steps.items():
        print(f"{key:<12} shape {arr.shape}  min {arr.min():.3f}  max {arr.max():.3f}  mean {arr.mean():.3f}")
    shape = X_photos.shape[1:]
    fig = plotting.plot_before_after_whitening(steps["unit"], steps["zca_display"], 0, shape)
    plotting.save(fig, "task1_1_zca")
    plt.show()

## Task 1.2 - Import somebody else's dataset (`vecs.npy`)

The file was saved with `np.save` on a dict, so it has to be loaded with `allow_pickle=True` and
unwrapped with `.item()`. Outer keys are `'1_pos'` ... `'9_pos'`, inner keys are the digits, and the
values are lists of 1024-dim vectors.

In [ ]:
vecs = multidigit_mnist.load_vecs()
print(multidigit_mnist.describe_vecs(vecs))

### 1.2.2 Two arrays (embeddings, labels) for an arbitrary position, in the file's order

In [ ]:
POSITION = 3   # change to any of 1..9

X_emb, y_emb = multidigit_mnist.arrays_for_position(vecs, POSITION)
print("embeddings:", X_emb.shape, X_emb.dtype)
print("labels    :", y_emb.shape, y_emb.dtype)
print("samples per digit:", np.bincount(y_emb))
print("first 15 labels in file order:", y_emb[:15])

### 1.2.3 Random permutation of data and labels (same permutation for both)

In [ ]:
X_shuf, y_shuf, perm = multidigit_mnist.shuffle_together(X_emb, y_emb)
print("first 15 labels after shuffle :", y_shuf[:15])
print("permutation starts with       :", perm[:15])
undo = np.argsort(perm)
print("pairs still matched:", np.array_equal(X_shuf[undo], X_emb) and np.array_equal(y_shuf[undo], y_emb))

# Part 2 - Nearest Neighbour classifier
Same setup as the lecturer's notebook: the first 500 training images are used as the training
set and every image is flattened to 784 pixels.

In [ ]:
Xtr, Ltr, X_test, L_test = mnist_data.load_mnist()
print("train:", Xtr.shape, Ltr.shape, Xtr.dtype, " test:", X_test.shape, L_test.shape)
plotting.plot_mnist_pair(Xtr, Ltr, 0, 1)
plt.show()

In [ ]:
num_sample = 500
Tr_set = mnist_data.flatten(Xtr[:num_sample])
Ltr_set = Ltr[:num_sample]
Test_images = mnist_data.flatten(X_test)
print("Tr_set:", Tr_set.shape, Tr_set.dtype, " Test_images:", Test_images.shape)

## Task 2.1.1 - run the provided 1-NN with L1 norm
`fix_dtype=False` runs the original code without any type conversion.

In [ ]:
nn_l1_original = NearestNeighbor(k=1, distance="l1", fix_dtype=False).train(Tr_set, Ltr_set)
pred = nn_l1_original.predict(Test_images)
print(nn_l1_original, "-> accuracy:", metrics.accuracy(L_test, pred))

## Task 2.1.2 - same classifier with the L2 norm

In [ ]:
nn_l2_original = NearestNeighbor(k=1, distance="l2", fix_dtype=False).train(Tr_set, Ltr_set)
pred = nn_l2_original.predict(Test_images)
print(nn_l2_original, "-> accuracy:", metrics.accuracy(L_test, pred))

## Task 2.1.3 - find and fix the bug

Comparing the distances from Python with ones calculated by hand shows the problem. MNIST pixels
are `uint8`, and for unsigned ints `3 - 5` wraps around to `254` instead of `-2`, so `np.abs(a - b)`
is wrong every time the training pixel is darker than the test pixel. For L2 the square overflows too.

Fix: cast to float before subtracting.

In [ ]:
a = np.array([3, 10, 200], dtype=np.uint8)
b = np.array([5, 10, 100], dtype=np.uint8)
print("uint8 a - b =", a - b)
print(distances.uint8_bug_demo(a, b))

# same check on real images: test image 0 vs the first 3 training images
for j in range(3):
    print(f"test image 0 vs train image {j}:", distances.uint8_bug_demo(Tr_set[j], Test_images[0]))

In [ ]:
results_1nn = {}
for dist in ("l1", "l2"):
    model = NearestNeighbor(k=1, distance=dist).train(Tr_set, Ltr_set)
    results_1nn[dist] = metrics.accuracy(L_test, model.predict(Test_images))
    print(model, "-> accuracy:", results_1nn[dist])

## Task 2.1.4 - k-NN
The k closest training images are found with `np.argpartition` and the most common label among
them wins (`np.bincount(...).argmax()`).

In [ ]:
knn = NearestNeighbor(k=3, distance="l2").train(Tr_set, Ltr_set)
pred_knn = knn.predict(Test_images)
print(knn, "-> accuracy:", metrics.accuracy(L_test, pred_knn), " macro F1:", round(metrics.macro_f1(L_test, pred_knn), 3))
fig = plotting.plot_confusion_matrix(metrics.confusion_matrix(L_test, pred_knn), [str(d) for d in range(10)], "3-NN, L2, 500 training images")
plotting.save(fig, "task2_1_confusion_3nn")
plt.show()

## Task 2.2 - choose k with 3-fold cross-validation (L2 norm)

The 500 training images are shuffled and split into 3 folds. For each k, every fold is used once
for validation and the other two for training, and the 3 accuracies are averaged. The test set
isn't used here.

In [ ]:
k_values = [1, 3, 5, 7, 9, 11, 15]
cv_results = cross_validation.cross_validate_k(Tr_set, Ltr_set, k_values, n_folds=3, distance="l2")

cv_table = pd.DataFrame({k: {"mean": m, **{f"fold {i+1}": f for i, f in enumerate(folds)}} for k, (m, folds) in cv_results.items()}).T
display(cv_table.round(3))
k_best = cross_validation.best_k(cv_results)
print("best k by 3-fold CV:", k_best)
fig = plotting.plot_cv_results(cv_results)
plotting.save(fig, "task2_2_cv")
plt.show()

### 2.2.2 Final, single evaluation on the test data with the best k

In [ ]:
final_knn = NearestNeighbor(k=k_best, distance="l2").train(Tr_set, Ltr_set)
test_accuracy = metrics.accuracy(L_test, final_knn.predict(Test_images))
print(f"{final_knn} -> test accuracy = {test_accuracy:.4f}")

# Part 3 - Support Vector Machines on Iris

Follows the *SVM clearly explained* blog: 80:20 split, `svm.SVC` with linear, poly and rbf kernels,
confusion matrices and a meshgrid decision boundary. For one-vs-one and one-vs-rest the same SVC
is wrapped in `OneVsOneClassifier` and `OneVsRestClassifier`.

In [ ]:
X_iris, y_iris, feature_names, iris_classes = svm_iris.load_iris()
X_train, X_test_iris, y_train, y_test_iris = svm_iris.split_80_20(X_iris, y_iris)
print("features:", feature_names)
print("train:", X_train.shape, " test:", X_test_iris.shape, " classes:", iris_classes)

## 3.1.1 Confusion matrices for one-vs-one and one-vs-rest, each kernel

In [ ]:
svm_models = svm_iris.train_all(X_train, y_train)
conf_matrices, svm_scores = svm_iris.evaluate_all(svm_models, X_test_iris, y_test_iris)
fig = plotting.plot_confusion_grid(conf_matrices, iris_classes)
plotting.save(fig, "task3_1_confusion_matrices")
plt.show()

## 3.1.2 Which kernel gives the best accuracy and F1?

In [ ]:
svm_table = pd.DataFrame(svm_scores)
display(svm_table.round(4))
print("best accuracy:", svm_iris.best_by(svm_scores, "accuracy"))
print("best F1      :", svm_iris.best_by(svm_scores, "f1_macro"))

## 3.1.3 Support vectors for each class (one-vs-rest)

Each binary "class c vs rest" SVC has its own support vectors, some from class c and some from
the other two classes.

In [ ]:
for kernel in svm_iris.KERNELS:
    sv = svm_iris.support_vectors_one_vs_rest(svm_models[("one-vs-rest", kernel)], X_train, y_train, iris_classes)
    print(f"--- kernel = {kernel}")
    for cls, info in sv.items():
        print(f"  {cls:<11} total {info['n_total']:>3}  from the class {info['n_from_class']:>3}  from the rest {info['n_from_rest']:>3}")

# the actual vectors for one case (linear kernel, class setosa)
sv_linear = svm_iris.support_vectors_one_vs_rest(svm_models[("one-vs-rest", "linear")], X_train, y_train, iris_classes)
print("\nsupport vectors of class setosa (linear, one-vs-rest):")
print(sv_linear["setosa"]["vectors_from_class"])

## 3.1.4 Decision boundaries for features 2 vs 3 and 3 vs 4

An SVM is trained on only the two features so the regions can be drawn in 2-D.

In [ ]:
fig, axes = plt.subplots(len(svm_iris.FEATURE_PAIRS), len(svm_iris.KERNELS), figsize=(15, 8))
for r, (title, pair) in enumerate(svm_iris.FEATURE_PAIRS.items()):
    for c, kernel in enumerate(svm_iris.KERNELS):
        model2d, X2 = svm_iris.train_on_feature_pair(X_train, y_train, pair, kernel=kernel)
        plotting.plot_decision_boundary(model2d, X2, y_train, [feature_names[i] for i in pair], f"{title} - {kernel}", ax=axes[r, c])
fig.tight_layout()
plotting.save(fig, "task3_1_decision_boundaries")
plt.show()

# Part 4 - Larger-scale systematic experimentation

Paper: Fernandez-Delgado et al. (2014), *Do we need hundreds of classifiers to solve real world
classification problems?*, JMLR 15. They test 179 classifiers on 121 UCI datasets; random forest
is best overall (about 82 % mean accuracy) with RBF SVM close behind.

Setup:
- 10 of the paper's datasets, downloaded from OpenML
- stratified 80/20 train/test split, new split for each of 5 seeds (42-46), mean and std reported
- imputation, one-hot encoding and scaling inside an sklearn `Pipeline`, so they are fitted on the
  training part only
- supervised: RBF SVM and random forest, hyperparameters from 5-fold `GridSearchCV` on the training part
- unsupervised: K-Means and Gaussian mixture with one cluster per class; each cluster gets the
  majority label of its training points
- metrics: accuracy (like the paper) and macro F1

In [ ]:
dataset_table = uci_experiments.describe_datasets()
display(dataset_table)

In [ ]:
runs = uci_experiments.run_all()  # takes a few minutes
runs.to_csv(config.LAB_DIR / "part4_runs.csv", index=False)
runs.head()

In [ ]:
summary = uci_experiments.summarise(runs)
display(summary.round(4))
print("Best model per dataset:")
display(uci_experiments.best_model_per_dataset(summary).round(4))
fig = plotting.plot_results_bar(summary, "accuracy_mean", "Mean test accuracy over 5 seeds")
plotting.save(fig, "task4_accuracy")
fig = plotting.plot_results_bar(summary, "f1_mean", "Mean macro F1 over 5 seeds")
plotting.save(fig, "task4_f1")
plt.show()

### Comparison with the paper
The paper's numbers go in `uci_experiments.DATASETS[name]["baseline"]`; `summarise` then fills in
`paper_baseline` and `diff_vs_paper`.

### Hyperparameters chosen by the grid search

In [ ]:
chosen = runs.dropna(subset=["best_params"])[["dataset", "model", "seed", "best_params"]]
display(chosen.head(20))